# Chapter 11: BI Without the Bloat

**Local-first Business Intelligence with Evidence.dev and Metabase**

This notebook demonstrates:
1. Setting up DuckDB for BI queries
2. Generating sample data for dashboards
3. Testing queries that will power Evidence reports
4. Exporting to SQLite for Metabase
5. Performance optimization with materialized views

## Setup

In [ ]:
import duckdb
import polars as pl
from pathlib import Path
from datetime import datetime, timedelta
import random

print("[OK] Imports successful")

## 1. Generate Sample E-commerce Data

In [ ]:
# Create data directory
Path('data/curated').mkdir(parents=True, exist_ok=True)

# Generate sample data
def generate_ecommerce_data(n_orders=10000, n_customers=1000, n_products=100):
    """Generate synthetic e-commerce data."""
    
    # Customers
    customers = pl.DataFrame({
        'customer_id': range(1, n_customers + 1),
        'name': [f'Customer {i}' for i in range(1, n_customers + 1)],
        'signup_date': [datetime(2023, 1, 1) + timedelta(days=random.randint(0, 365)) for _ in range(n_customers)],
        'updated_at': [datetime.now()] * n_customers
    })
    
    # Products
    categories = ['Electronics', 'Clothing', 'Home & Garden']
    products = pl.DataFrame({
        'product_id': range(1, n_products + 1),
        'product_name': [f'Product {i}' for i in range(1, n_products + 1)],
        'category': [random.choice(categories) for _ in range(n_products)],
        'price': [round(random.uniform(10, 500), 2) for _ in range(n_products)],
        'updated_at': [datetime.now()] * n_products
    })
    
    # Orders
    start_date = datetime(2024, 1, 1)
    orders = pl.DataFrame({
        'order_id': range(1, n_orders + 1),
        'customer_id': [random.randint(1, n_customers) for _ in range(n_orders)],
        'order_date': [start_date + timedelta(days=random.randint(0, 300)) for _ in range(n_orders)],
        'total_amount': [round(random.uniform(20, 1000), 2) for _ in range(n_orders)],
        'updated_at': [datetime.now()] * n_orders
    })
    
    # Order items
    order_items = []
    for order_id in range(1, n_orders + 1):
        n_items = random.randint(1, 5)
        for _ in range(n_items):
            product_id = random.randint(1, n_products)
            quantity = random.randint(1, 3)
            order_items.append({
                'order_id': order_id,
                'product_id': product_id,
                'quantity': quantity,
                'total_amount': round(random.uniform(10, 300), 2)
            })
    
    order_items_df = pl.DataFrame(order_items)
    # Add order_date from orders
    order_items_df = order_items_df.join(
        orders.select(['order_id', 'order_date']),
        on='order_id'
    )
    
    return customers, products, orders, order_items_df

customers, products, orders, order_items = generate_ecommerce_data()

print(f"Generated:")
print(f"  Customers: {len(customers):,}")
print(f"  Products: {len(products):,}")
print(f"  Orders: {len(orders):,}")
print(f"  Order items: {len(order_items):,}")

## 2. Create DuckDB Database

In [ ]:
# Connect to DuckDB
con = duckdb.connect('data/curated/analytics.duckdb')

# Create schema
con.execute("CREATE SCHEMA IF NOT EXISTS curated")

# Write tables
con.execute("CREATE OR REPLACE TABLE curated.customers AS SELECT * FROM customers")
con.execute("CREATE OR REPLACE TABLE curated.products AS SELECT * FROM products")
con.execute("CREATE OR REPLACE TABLE curated.orders AS SELECT * FROM orders")
con.execute("CREATE OR REPLACE TABLE curated.order_items AS SELECT * FROM order_items")

print("[OK] DuckDB database created at data/curated/analytics.duckdb")

## 3. Test Revenue Dashboard Queries

In [ ]:
# Monthly revenue (Evidence query)
monthly_revenue = con.execute("""
    SELECT
        DATE_TRUNC('month', order_date) as month,
        SUM(total_amount) as revenue,
        COUNT(DISTINCT customer_id) as customers,
        SUM(total_amount) / COUNT(DISTINCT customer_id) as avg_customer_value
    FROM curated.orders
    WHERE order_date >= CURRENT_DATE - INTERVAL 12 MONTH
    GROUP BY 1
    ORDER BY 1
""").df()

print("Monthly Revenue:")
print(monthly_revenue)

In [ ]:
# Recent metrics (Evidence query)
recent_metrics = con.execute("""
    SELECT
        SUM(total_amount) as total_revenue,
        COUNT(*) as total_orders,
        SUM(total_amount) / COUNT(*) as avg_order_value
    FROM curated.orders
    WHERE order_date >= CURRENT_DATE - INTERVAL 30 DAY
""").df()

print("\nLast 30 Days:")
print(recent_metrics)

In [ ]:
# Top products (Evidence query)
top_products = con.execute("""
    SELECT
        product_name,
        SUM(quantity) as units_sold,
        SUM(oi.total_amount) as revenue
    FROM curated.order_items oi
    JOIN curated.products p USING (product_id)
    WHERE order_date >= CURRENT_DATE - INTERVAL 30 DAY
    GROUP BY 1
    ORDER BY revenue DESC
    LIMIT 10
""").df()

print("\nTop 10 Products:")
print(top_products)

## 4. Test Data Quality Dashboard Queries

In [ ]:
# Pipeline health
pipeline_stats = con.execute("""
    SELECT
        table_name,
        MAX(updated_at) as last_update,
        COUNT(*) as row_count,
        ROUND((CURRENT_TIMESTAMP - MAX(updated_at)) / 3600, 1) as hours_since_update
    FROM (
        SELECT 'orders' as table_name, updated_at FROM curated.orders
        UNION ALL
        SELECT 'customers', updated_at FROM curated.customers
        UNION ALL
        SELECT 'products', updated_at FROM curated.products
    ) t
    GROUP BY table_name
""").df()

print("Pipeline Health:")
print(pipeline_stats)

In [ ]:
# Quality checks
quality_metrics = con.execute("""
    SELECT
        'Null customer_ids' as check_name,
        COUNT(*) as failures
    FROM curated.orders
    WHERE customer_id IS NULL

    UNION ALL

    SELECT
        'Negative prices',
        COUNT(*)
    FROM curated.products
    WHERE price < 0

    UNION ALL

    SELECT
        'Future order dates',
        COUNT(*)
    FROM curated.orders
    WHERE order_date > CURRENT_DATE
""").df()

print("\nQuality Checks:")
print(quality_metrics)

if quality_metrics['failures'].sum() > 0:
    print("\n[WARNING] Quality issues detected!")
else:
    print("\n[OK] All quality checks passed")

## 5. Materialize Daily Metrics (Performance Optimization)

In [ ]:
# Create materialized daily metrics table
con.execute("""
    CREATE OR REPLACE TABLE curated.daily_metrics AS
    SELECT
        DATE_TRUNC('day', order_date) as date,
        SUM(total_amount) as revenue,
        COUNT(*) as orders,
        COUNT(DISTINCT customer_id) as active_customers
    FROM curated.orders
    GROUP BY 1
""")

print("[OK] Daily metrics materialized")

# Verify
daily_metrics = con.execute("SELECT * FROM curated.daily_metrics ORDER BY date DESC LIMIT 10").df()
print("\nDaily Metrics (last 10 days):")
print(daily_metrics)

## 6. Export to SQLite for Metabase

In [ ]:
# Export DuckDB to SQLite format
export_dir = 'data/curated/metabase_export'
Path(export_dir).parent.mkdir(parents=True, exist_ok=True)

con.execute(f"EXPORT DATABASE '{export_dir}' (FORMAT SQLITE);")

print(f"[OK] Exported to {export_dir}")
print(f"[INFO] In Metabase, connect to: {export_dir}.db")

## 7. Performance Comparison: Raw vs Materialized

In [ ]:
import time

# Query raw orders
start = time.time()
con.execute("""
    SELECT
        DATE_TRUNC('day', order_date) as date,
        SUM(total_amount) as revenue,
        COUNT(*) as orders
    FROM curated.orders
    GROUP BY 1
""").fetchall()
raw_time = time.time() - start

# Query materialized view
start = time.time()
con.execute("""
    SELECT date, revenue, orders
    FROM curated.daily_metrics
""").fetchall()
materialized_time = time.time() - start

print(f"Performance Comparison:")
print(f"  Raw query: {raw_time*1000:.2f}ms")
print(f"  Materialized query: {materialized_time*1000:.2f}ms")
print(f"  Speedup: {raw_time/materialized_time:.1f}x")

## 8. Category Drilldown (Parameterized Query)

In [ ]:
# Test parameterized query (used in Evidence dropdown)
category = 'Electronics'

category_sales = con.execute(f"""
    SELECT
        DATE_TRUNC('week', order_date) as week,
        SUM(oi.total_amount) as revenue
    FROM curated.order_items oi
    JOIN curated.products p USING (product_id)
    WHERE p.category = '{category}'
      AND order_date >= CURRENT_DATE - INTERVAL 90 DAY
    GROUP BY 1
    ORDER BY 1
""").df()

print(f"Weekly Sales for {category}:")
print(category_sales)

## Summary

You've:
- ✅ Generated sample e-commerce data
- ✅ Created DuckDB database
- ✅ Tested all Evidence dashboard queries
- ✅ Tested data quality checks
- ✅ Materialized daily metrics for performance
- ✅ Exported to SQLite for Metabase

**Next steps:**
1. Run Evidence: `cd evidence && npm run dev`
2. Start Metabase: `docker compose -f docker/compose.yaml up -d metabase`
3. Access dashboards at http://localhost:3000 (Evidence) and http://localhost:3001 (Metabase)

**Cost: $0** (vs $400/month for traditional BI)